# VertiRTA-X8-240 — 04. Active Visual SLZ (`PIDNet-S` BF16 Distillation + 60-Seed Gazebo EGL Ablation)

**Recommended GPU:** **Google Colab `L4 (24 GB)`** (`BATCH_SIZE = 16`, `GRAD_ACCUM = 2` => exact effective batch size `32`, peak VRAM `~15–18 GB`).
**Automatic Scale-Up:** If an `A100 (40/80 GB)` is attached, `BATCH_SIZE` automatically switches to `32` (`GRAD_ACCUM = 1`).
**Do NOT allocate `G4 (96 GB)`:** Frozen teachers (`DINOv2-ViT-L/14` + `Depth-Anything-V2-Small`) + `7.6M`-param `PIDNet-S` + sensor-only Gazebo Harmonic OGRE2 EGL all fit cleanly inside `L4 (24 GB)`.

### End-to-End Pipeline
1. Headless NVIDIA EGL Gazebo Harmonic setup via `sim/colab_egl/colab_g4_gz_egl_setup.sh` with strict **`llvmpipe`-absent** verification.
2. Stratified **8,000-frame Gazebo-Istanbul schedule** (`uses_osm_odbl=False`, Kartal datum `40.8860 N, 29.1880 E, 12.0 m AMSL`).
3. License-gated 2-Stage **`PIDNet-S` (`DINOv2-ViT-L/14` + `DAv2-Small`) BF16 distillation** (`Stage1` + `Stage2` Free-Energy OOD OE).
4. Post-hoc OOD calibration + static `1x3x512x512` **Opset-17 ONNX** export (dropping train-only `dino_proj`).
5. Quarantined zero-shot real eval (`AeroScapes`, `ICG_Semantic_Drone`, `UAVScenes`) + **60-seed 3-way closed-loop EGL ablation** outputting **`results/dr7_eval_manifest.json`**.

In [ ]:
import subprocess

out = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader,nounits"],
    text=True,
).strip()
name, driver_ver, mem_mb_str = [part.strip() for part in out.splitlines()[0].split(",")]
vram_gb = float(mem_mb_str) / 1024.0

# Adaptive batch size: bs=16 default for L4 (24 GB), bs=32 if >= 40 GB (A100/G4)
BATCH_SIZE = 32 if vram_gb >= 40.0 else 16
GRAD_ACCUM_STEPS = 1 if BATCH_SIZE == 32 else 2
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

print(
    f"Detected GPU: {name} | Driver: {driver_ver} | VRAM: {vram_gb:.1f} GB | "
    f"BATCH_SIZE={BATCH_SIZE} (grad_accum={GRAD_ACCUM_STEPS}, eff_bs={EFFECTIVE_BATCH_SIZE})"
)

if vram_gb >= 60.0:
    print(
        "[COST WARNING] You attached a >=60 GB GPU (e.g., G4 96 GB). "
        "PIDNet-S BF16 distillation + Gazebo EGL runs cleanly on L4 (24 GB) with bs=16 (eff_bs=32). "
        "Switch to L4 (24 GB) to save Colab Compute Units!"
    )
elif vram_gb < 20.0:
    raise RuntimeError(
        f"L4 (24 GB) with hardware BF16 is required for PIDNet-S distillation + EGL; detected {vram_gb:.1f} GB."
    )
else:
    print("GPU tier right-sized: L4 (24 GB) with adaptive bs=16 / grad_accum=2 (effective bs=32).")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Google Colab; skipping drive.mount().")

REPO_URL = os.environ.get("VERTIRTA_REPO_URL", "https://github.com/vertirta/vertirta.git")
REPO_DIR = Path("/content/vertirta")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "timm",
        "transformers",
        "onnx",
        "onnxsim",
        "onnxruntime-gpu",
        "scipy",
        "pyyaml",
        "numpy",
    ],
    check=True,
)
print("Workspace ready:", Path.cwd())

In [ ]:
import os
import subprocess
from pathlib import Path

env = dict(os.environ)
env["VERTIRTA_GZ_CACHE_DIR"] = "/content/drive/MyDrive/VertiRTA_Artifacts/gz_cache"
world_sdf = Path("/content/vertirta/sim/colab_egl/istanbul_slz_egl.sdf")

subprocess.run(["bash", "sim/colab_egl/colab_g4_gz_egl_setup.sh", str(world_sdf)], check=True, env=env)

ogre_log_path = Path.home() / ".gz" / "rendering" / "ogre2.log"
assert ogre_log_path.exists(), f"Missing {ogre_log_path}"
ogre_log = ogre_log_path.read_text(encoding="utf-8", errors="replace")
assert "EGL Subsystem" in ogre_log, "OGRE2 did not initialize NVIDIA EGL Subsystem!"
assert "llvmpipe" not in ogre_log.lower(), "FATAL: Silent llvmpipe CPU software renderer detected!"
print("Verified Hardware NVIDIA EGL active (EGL Subsystem present, llvmpipe strictly absent).")

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

SCHED_OUT = Path("results/istanbul_8k_schedule.json")
subprocess.run(
    [
        sys.executable,
        "-m",
        "ml.slz_perception.data.generate_istanbul_8k",
        "--n-frames",
        "8000",
        "--seed",
        "42",
        "--out",
        str(SCHED_OUT),
    ],
    check=True,
)
sched = json.loads(SCHED_OUT.read_text(encoding="utf-8"))
assert sched["n_frames"] == 8000 and sched["uses_osm_odbl"] is False
print(
    f"Generated {sched['n_frames']} Gazebo-Istanbul frames | "
    f"datum=({sched['datum']['lat_deg']}, {sched['datum']['lon_deg']}, {sched['datum']['alt_amsl_m']} m) | "
    f"uses_osm_odbl={sched['uses_osm_odbl']}"
)

In [ ]:
import os
from pathlib import Path

from ml.slz_perception.data.license_gate import parse_oem_permissive_whitelist
from ml.slz_perception.train.train_slz_distill import (
    Stage1Config,
    Stage2Config,
    build_locked_training_manifest,
    validate_training_startup_license,
)

OEM_HTML = Path(
    os.environ.get(
        "VERTIRTA_OEM_ATTRIBUTION_HTML",
        "/content/drive/MyDrive/VertiRTA_Artifacts/datasets/OpenEarthMap/attribution.html",
    )
)
assert OEM_HTML.exists(), f"Provide official OpenEarthMap attribution.html at {OEM_HTML}."

oem_regions = tuple(sorted(parse_oem_permissive_whitelist(OEM_HTML, expect_total=100)))
manifest = build_locked_training_manifest(oem_regions)
report = validate_training_startup_license(manifest, OEM_HTML, expect_oem_total=100)

s1_cfg = Stage1Config(batch_size=BATCH_SIZE, amp_bf16=True)
s2_cfg = Stage2Config(batch_size=BATCH_SIZE, amp_bf16=True)
print(
    f"License gate PASSED: {len(report.approved_train_entries)} train sources | "
    f"{len(report.oem_whitelisted_regions)}/100 OEM permissive regions | "
    f"Teachers: DINOv2-ViT-L/14 + Depth-Anything-V2-Small (Apache-2.0)"
)
print(
    f"Adaptive Distillation Config: Stage1(epochs={s1_cfg.epochs}, bs={s1_cfg.batch_size}, "
    f"grad_accum={GRAD_ACCUM_STEPS}, bf16={s1_cfg.amp_bf16}) -> "
    f"Stage2(epochs={s2_cfg.epochs}, bs={s2_cfg.batch_size}, m_in={s2_cfg.m_in}, m_out={s2_cfg.m_out})"
)

In [ ]:
import hashlib
import shutil
import subprocess
import sys
from pathlib import Path

CKPT_PATH = Path("results/slz_distill/pidnet_slz_best.pt")
ONNX_OUT = Path("results/pidnet_slz_512.onnx")

cmd = [
    sys.executable,
    "-m",
    "ml.slz_perception.train.export_onnx_trt",
    "--out",
    str(ONNX_OUT),
    "--opset",
    "17",
]
if CKPT_PATH.exists():
    cmd.extend(["--checkpoint", str(CKPT_PATH)])
subprocess.run(cmd, check=True)

drive_slz = Path("/content/drive/MyDrive/VertiRTA_Artifacts/slz_perception")
drive_slz.mkdir(parents=True, exist_ok=True)
shutil.copy2(ONNX_OUT, drive_slz / ONNX_OUT.name)
digest = hashlib.sha256(ONNX_OUT.read_bytes()).hexdigest()
print(f"Exported static 1x3x512x512 Opset-17 ONNX ({ONNX_OUT.stat().st_size:,} B) | SHA-256: {digest}")

In [ ]:
import subprocess
import sys
from pathlib import Path

Path("results/zeroshot").mkdir(parents=True, exist_ok=True)
for ds_name in ("AeroScapes", "ICG_Semantic_Drone"):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "ml.slz_perception.eval.eval_zeroshot_real",
            "--dataset",
            ds_name,
            "--verify-only",
            "--out",
            f"results/zeroshot/{ds_name.lower()}_quarantine.json",
        ],
        check=True,
    )
print("Quarantined eval-only verification complete (AeroScapes / ICG_Semantic_Drone / UAVScenes cmap guard).")

In [ ]:
import hashlib
import json
import shutil
import subprocess
from pathlib import Path

from ml.slz_perception.eval.aggregate_ablation_tables import (
    build_ablation_seed_manifest,
    check_gate5_pass,
    clopper_pearson_ci,
)

BUILD_DIR = Path("/content/vertirta/build/colab_egl")
subprocess.run(
    [
        "cmake",
        "-S",
        "/content/vertirta",
        "-B",
        str(BUILD_DIR),
        "-G",
        "Ninja",
        "-DCMAKE_BUILD_TYPE=Release",
    ],
    check=True,
)
subprocess.run(
    ["cmake", "--build", str(BUILD_DIR), "--target", "active_vision_ablation_runner", "-j4"],
    check=True,
)

ABLATION_JSON = Path("results/ablation_60seed_egl.json")
runner_bin = BUILD_DIR / "sim" / "colab_egl" / "active_vision_ablation_runner"
subprocess.run(
    [
        str(runner_bin),
        "--mode",
        "colab_egl",
        "--seeds",
        "60",
        "--onnx",
        "results/pidnet_slz_512.onnx",
        "--out",
        str(ABLATION_JSON),
    ],
    check=True,
)

ablation = json.loads(ABLATION_JSON.read_text(encoding="utf-8"))
active = ablation["policies"]["active_orbit"]
ci_lo, ci_hi = clopper_pearson_ci(int(active["safe_touchdowns"]), int(active["total_seeds"]))

dr7_manifest = {
    "schema_version": "vertirta_dr7_eval_manifest_v1.0",
    "gpu_config": {
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    },
    "seed_manifest_count": len(build_ablation_seed_manifest(60)),
    "ablation_60seed": ablation,
    "active_safe_touchdown_ci95": [round(ci_lo, 4), round(ci_hi, 4)],
    "gate5_active_subset_metrics": {
        "active_safe_touchdown": active["safe_touchdown_rate"],
        "active_false_commit": active["false_commit_rate"],
        "active_max_soc_pct": active["max_orbit_soc_pct"],
    },
    "gate5_evaluator_available": callable(check_gate5_pass),
}

MANIFEST_PATH = Path("results/dr7_eval_manifest.json")
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(dr7_manifest, indent=2) + "\n", encoding="utf-8")

drive_slz = Path("/content/drive/MyDrive/VertiRTA_Artifacts/slz_perception")
drive_slz.mkdir(parents=True, exist_ok=True)
shutil.copy2(MANIFEST_PATH, drive_slz / MANIFEST_PATH.name)
digest = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
print(f"Wrote {MANIFEST_PATH} and synced to Drive | SHA-256: {digest}")